# Hammurab.AI — Qwen2.5-1.5B LoRA Fine-Tuning
**CIF425 Term Project**

Bu notebook Qwen2.5-1.5B modelini hukuk Q&A verisiyle LoRA ile fine-tune eder.

**Kullanim:** Runtime > Change runtime type > **T4 GPU** sec, sonra **Run All**

## 1. Repo'yu Klonla & Bagimliliklari Kur

In [ ]:
!git clone https://github.com/Alp33er/hammurab.ai.git
%cd hammurab.ai
!git checkout development

In [ ]:
!pip install -q transformers torch datasets accelerate peft bitsandbytes

## 2. GPU Kontrol

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Veri Seti Olustur
Kanun JSON'larindan 18K+ Q&A cifti uretir.

In [ ]:
!python training/prepare_dataset.py

In [ ]:
# Veri seti ornegi
import json
with open("training/data/hukuk_qa.jsonl", "r") as f:
    sample = json.loads(f.readline())
print("PROMPT:")
print(sample["prompt"][:500])
print("\nCOMPLETION:")
print(sample["completion"][:300])

## 4. Fine-Tuning (LoRA)
Qwen2.5-1.5B + 4-bit quantization + LoRA

~30-45 dakika (T4 GPU)

In [ ]:
!python training/fine_tune.py

## 5. Modeli Test Et

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("./model", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("./model", trust_remote_code=True, torch_dtype=torch.float16).to("cuda")
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def sor(soru, mevzuat=""):
    prompt = "<|im_start|>system\nSen Turk hukuku konusunda uzman bir hukuk asistanisin.<|im_end|>\n"
    user_content = soru
    if mevzuat:
        user_content += f"\n\nIlgili Mevzuat:\n{mevzuat}"
    prompt += f"<|im_start|>user\n{user_content}<|im_end|>\n"
    prompt += "<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    for stop in ["<|im_start|>", "<|im_end|>"]:
        if stop in response:
            response = response[:response.index(stop)]
    return response.strip()

print("Model hazir!")

In [ ]:
# Test 1
print(sor("Is kazasi durumunda iscinin haklari nelerdir?"))

In [ ]:
# Test 2
print(sor("Turk Borclar Kanunu madde 49 ne diyor?"))

In [ ]:
# Test 3
print(sor("Kidem tazminati nasil hesaplanir?"))

## 6. Modeli Indir

In [ ]:
!zip -r /content/hammurab_model.zip model/

from google.colab import files
files.download("/content/hammurab_model.zip")

## 7. (Alternatif) Google Drive'a Kaydet

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!cp -r model/ /content/drive/MyDrive/hammurab_model/
print("Model Google Drive'a kaydedildi!")